## Instalação e Importação de Bibliotecas

In [3]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="keras")

In [4]:
#
# Instalação das bibliotecas para o fluxo TensorFlow Hub
!pip install -U tensorflow-text keras_hub keras

  Using cached tensorflow_text-2.21.1-cp313-cp313-manylinux_2_27_x86_64.whl.metadata (2.0 kB)
  Using cached tensorflow-2.21.0-cp313-cp313-manylinux_2_27_x86_64.whl.metadata (4.4 kB)


In [5]:
import os
import shutil
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sb

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
import tensorflow_text as text

import keras
import keras_hub

%matplotlib inline
import IPython
import sys

tf.get_logger().setLevel('ERROR')

print("Bibliotecas configuradas.")

Bibliotecas configuradas.


### Verificação da instalação

In [6]:
# Verificação das versões das bibliotecas Python importadas
print("Versão do Python: ",sys.version)
print("Versão do IPython: ",IPython.__version__)
print("Versão do NumPy: ",np.__version__)
print("Versão do Pandas: ",pd.__version__)
print("Versão do Matplotlib: ",matplotlib.__version__)
print("Versão do Seaborn: ",sb.__version__)
print("Versão do Sciki-Learn: ",sklearn.__version__)
print("Versão do TensorFlow: ",tf.__version__)
print("Versão do Keras:",keras.__version__)
print("Versão do Keras-hub: ",keras.__version__)
#print("Versão do TensorFlow-Hub: ",hub.__version__)
print("Versão do TensorFlow-Text: ",text.__version__)


Versão do Python:  3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Versão do IPython:  7.34.0
Versão do NumPy:  2.1.3
Versão do Pandas:  2.2.3
Versão do Matplotlib:  3.10.0
Versão do Seaborn:  0.13.2
Versão do Sciki-Learn:  1.6.1
Versão do TensorFlow:  2.20.0
Versão do Keras: 3.15.1
Versão do Keras-hub:  3.15.1
Versão do TensorFlow-Text:  2.20.1


## Configurando um estado aleatório

In [7]:
import os

# Definindo um germe aleatório
GERME_ALEATORIO = 42

# Defini um estado aleatório para Python
os.environ['PYTHONHASHSEED'] = str(GERME_ALEATORIO)

# Defini um estado aleatório para Python random
import random
random.seed(GERME_ALEATORIO)

# Determinar um estado aleatório para Python
os.environ['PYTHONHASHSEED'] = str(GERME_ALEATORIO)

# Determinar um estado aleatório para Python random
import random
random.seed(GERME_ALEATORIO)

# Determinar um estado aleatório para NumPy
import numpy as np
np.random.seed(GERME_ALEATORIO)
# Definindo um estado aleatório para Keras
from tensorflow import keras
keras.utils.set_random_seed(GERME_ALEATORIO)
# Definindo um estado aleatório para TensorFlow
import tensorflow as tf
tf.random.set_seed(GERME_ALEATORIO)

os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

print("Germe aleatório fixo")

Germe aleatório fixo


## Carregamento e preparação da base de dados

In [8]:
df = pd.read_csv("/content/scamsdataset_pt_clean.csv")

# Repartição estratificada: 0.70 treino e 0.30 temporário (validação e teste)
X_train, X_temp, y_train, y_temp = train_test_split(
    df['texto'], df['classe'], test_size=0.3, random_state=GERME_ALEATORIO, stratify=df['classe']
)
# Validação 0.15 e teste 0.15
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=GERME_ALEATORIO, stratify=y_temp
)

### Converção para tensores - tf.data.Dataset

 - Ajuste dos pesos para corregir o desbalanceamento

  - Configuração dos pesos das classes

In [17]:
# 'legitimas' -> 0, 'smishing' -> 1
label_lookup = tf.keras.layers.StringLookup(
    vocabulary=['Legitimate', 'Smishing'],
    num_oov_indices=0
)

# Calculo dos pesos para balanceamento das classes - compensação do desbalanceamento
y_train_num= label_lookup(y_train.values).numpy()
pesos = compute_class_weight('balanced', classes=np.unique(y_train_num), y=y_train_num)
pesos_dict = dict(enumerate(pesos))
print(f"Pesos aplicados: {pesos_dict}")

batch_size = 64
AUTOTUNE = tf.data.AUTOTUNE

# Mapeia a string para número dentro do tf.data
def encode_dataset(x, y, is_train=False):
  ds = tf.data.Dataset.from_tensor_slices((x, y))
  # Transforma o label em string para númerico
  ds = ds.map(lambda text, label: (text, tf.cast(label_lookup(label), tf.float32)), num_parallel_calls=AUTOTUNE)
  if is_train:
    return ds.batch(batch_size).cache().prefetch(AUTOTUNE)
  return ds.batch(batch_size)

treino_ds = encode_dataset(X_train, y_train, is_train=True)
val_ds = encode_dataset(X_val, y_val, is_train=True)
teste_ds = encode_dataset(X_test, y_test, is_train=False)

Pesos aplicados: {0: np.float64(0.8463726884779517), 1: np.float64(1.2217659137577002)}


## Carregando modelos pré-treinados do Keras Hub

In [18]:
# Construção do modelo
def build_classifier_model():
  # Carregando a arquitetura, pesos, processador e classificador
  modelo = keras_hub.models.DistilBertTextClassifier.from_preset(
      "distil_bert_base_multi",
      num_classes=1,
      activation="sigmoid"
  )
  # Congelamento dos pesos da rede
  modelo.backbone.trainable = False
  return modelo

modelo= build_classifier_model()

### Verificando se o modelo é executado com a saída do modelo de pré-processamento

- Visualização da estrutura do modelo

In [19]:
# Verifica se o modelo é executado com a saída do modelo de pré-processamento
print("--- Testando saída do modelo ---")
# Instanciando o pré-processador
preprocessor = keras_hub.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_multi"
)
preprocessed_text = preprocessor(tf.constant(["Este é um teste para verificar o fluxo de tensores."]))
# Passando o texto pré-processado para o modelo
bert_raw_result = modelo(preprocessed_text)
print(f"Probabilidade calculada (Tensor): {bert_raw_result.numpy()}\n")

--- Testando saída do modelo ---
Probabilidade calculada (Tensor): [[0.4920964]]



### Exibindo a estrutura do modelo

In [20]:
print("Estrutura do modelo:")
modelo.summary()

Estrutura do modelo:


Preprocessor: "distil_bert_text_classifier_preprocessor_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ distil_bert_tokenizer (DistilBertTokenizer)                   │                      Vocab size: 119,547 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "distil_bert_text_classifier_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ distil_bert_backbone          │ (None, None, 768)         │     134,734,080 │ padding_mask[0][0],        │
│ (DistilBertBackbone)          │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ get_item_1 (GetItem)          │ (None, 768)               │               0 │ distil_bert_backbone[0][0] │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pooled_dense (Dense)          │ (None, 768)               │         590,592 │ get_item_1[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ output_dropout (Dropout)      │ (None, 768)               │               0 │ pooled_dense[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ logits (Dense)                │ (None, 1)                 │             769 │ output_dropout[0][0]       │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 135,325,441 (516.23 MB)

 Trainable params: 591,361 (2.26 MB)

 Non-trainable params: 134,734,080 (513.97 MB)

# Treinamento do modelo & Fine-Tuning

#### Definição da losses function e métricas

In [21]:
loss = tf.keras.losses.BinaryCrossentropy()
metrics = [tf.keras.metrics.BinaryAccuracy(name='acuracy')]

#### Configurações do Otimizador

In [22]:
# Config steps
epochs = 5
init_lr = 3e-5 # BERT geralmente performa melhor com taxas menores como 3e-5 ou 5e-5
steps_per_epoch = tf.data.experimental.cardinality(treino_ds).numpy()
num_train_steps = steps_per_epoch * epochs
num_warmup_steps = int(0.1 * num_train_steps)

In [23]:
# Agendador nativo de taxa de aprendizagem - warmup + linear decay
class WarmupLinearDecaySchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, initial_learning_rate, warmup_steps, total_steps):
      super().__init__()
      self.initial_learning_rate = float(initial_learning_rate)
      self.warmup_steps = float(warmup_steps)
      self.total_steps = float(total_steps)

    def __call__(self, step):
      step = tf.cast(step, tf.float32)
      warmup_steps = tf.cast(self.warmup_steps, tf.float32)
      total_steps = tf.cast(self.total_steps, tf.float32)

      # Incremento linear ao longo do warmup
      warmup_lr = self.initial_learning_rate * (step / tf.maximum(1.0, warmup_steps))
      decay_steps = tf.maximum(1.0, total_steps - warmup_steps)
      decay_lr = tf.maximum(0.0, self.initial_learning_rate)

      return tf.where(step < warmup_steps, warmup_lr, decay_lr)

  # Instanciando agendador e otimizador
lr_schedule = WarmupLinearDecaySchedule(
    initial_learning_rate=init_lr,
    warmup_steps=num_warmup_steps,
    total_steps=num_train_steps
)
optimizer = tf.keras.optimizers.AdamW(learning_rate=lr_schedule)

#### Compilação e Treinamento

In [24]:
#metrics = [
#    tf.keras.metrics.BinaryAccuracy(name='accuracy'),
#    tf.keras.metrics.Precision(name='precision'),
#    tf.keras.metrics.Recall(name='recall'),
#]



modelo.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=metrics
)

print(f"Iniciando treinamento (steps totais: {num_train_steps}, warmup: {num_warmup_steps})...")
history = modelo.fit(
    x=treino_ds,
    validation_data=val_ds,
    epochs=epochs,
    class_weight=pesos_dict
)

Iniciando treinamento (steps totais: 190, warmup: 19)...
Epoch 1/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 75s 2s/step - acuracy: 0.6286 - loss: 0.6859 - val_acuracy: 0.8431 - val_loss: 0.6691
Epoch 2/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 68s 2s/step - acuracy: 0.7920 - loss: 0.6591 - val_acuracy: 0.8549 - val_loss: 0.6377
Epoch 3/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 68s 2s/step - acuracy: 0.8298 - loss: 0.6296 - val_acuracy: 0.8706 - val_loss: 0.6026
Epoch 4/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 67s 2s/step - acuracy: 0.8517 - loss: 0.5977 - val_acuracy: 0.8706 - val_loss: 0.5669
Epoch 5/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 68s 2s/step - acuracy: 0.8559 - loss: 0.5611 - val_acuracy: 0.8706 - val_loss: 0.5298
